In [ ]:
%%capture
# Reinstall torch + torchvision TOGETHER from PyTorch's CUDA 12.4 index
# This ensures the C++ binaries match each other
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Install training libraries
!pip install "trl>=0.15.0" peft accelerate datasets

# Verify
import torch, torchvision
print(f"torch {torch.__version__}, torchvision {torchvision.__version__}, CUDA {torch.cuda.is_available()}")


In [29]:
import requests
from datasets import Dataset

# Load Dataset
dataset_url = "https://raw.githubusercontent.com/archittmittal/MetaxBangalore/main/training_prompts.json"
data = requests.get(dataset_url).json()
train_dataset = Dataset.from_list(data)

def format_for_qwen(example):
    # This ChatML format tells Qwen: "Listen to the system prompt!"
    system_msg = "<|im_start|>system\nYou are an Elite Executive Assistant. ALWAYS use the following format:\n<thought>\n[reasoning]\n</thought>\n{\"command\": \"...\", \"parameters\": {...}}<|im_end|>\n"
    user_msg = f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
    assistant_start = "<|im_start|>assistant\n<thought>\n"
    return {"prompt": system_msg + user_msg + assistant_start}

train_dataset = train_dataset.map(format_for_qwen)
print("✅ Cell 2: Model set to ChatML 'Instruction' Mode.")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

✅ Cell 2: Model set to ChatML 'Instruction' Mode.


In [30]:
import re, json

def reward_structure(completions, **kwargs):
    if len(completions) > 0:
        # Show us what it says and how it's scored
        print(f"\n[MODEL]: {completions[0][:100]}...")
    
    rewards = []
    for c in completions:
        score = 0.0
        if "</thought>" in c: score += 15.0 # HUGE REWARD
        if "{" in c and "}" in c: score += 15.0 # HUGE REWARD
        if "[" in c and "{" not in c: score -= 5.0 # PENALTY for wrong format
        rewards.append(float(score))
        
    if len(rewards) > 0: print(f"   [FORMAT SCORE]: {rewards[0]}")
    return rewards

def reward_reasoning_quality(completions, **kwargs):
    rewards = []
    for c in completions:
        txt = c.lower()
        score = 0.0
        keywords = ["priority", "flight", "fixed", "social", "resolve", "deadline"]
        for word in keywords:
            if word in txt: score += 0.5
        rewards.append(float(score))
    return rewards

print("✅ Cell 3: Rewards set to High-Stakes Mode.")


✅ Cell 3: Rewards set to High-Stakes Mode.


In [31]:
# Final Hyper-Training Configuration
hyper_args = GRPOConfig(
    output_dir="/kaggle/working/output",
    learning_rate=1e-4, # Higher learning rate for 1.5B model
    lr_scheduler_type="cosine",
    warmup_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4, # Use 4 for better math/learning
    max_completion_length=400,
    temperature=0.9, 
    num_train_epochs=1,
    logging_steps=1,
    save_steps=200,
    fp16=True,
    push_to_hub=True,
    hub_model_id="purvansh01/conflict-env-qwen-grpo-v2",
    hub_token=hf_token,
)

hyper_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_structure, reward_reasoning_quality],
    args=hyper_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("🚀 Cell 4: Starting Final Training...")
hyper_trainer.train()


🚀 Cell 4: Starting Final Training...

[MODEL]: The current scenario involves a complex mix of requests that need to be prioritized while considerin...
   [FORMAT SCORE]: 30.0


Step,Training Loss
1,-0.010812
2,0.132359
3,-0.060514
4,-0.062382
5,0.166595
6,0.398016
7,-0.086515
8,0.173245
9,-0.061106
10,-0.191713



[MODEL]: To resolve the scheduling conflict, I need to consider the priorities of the various attendees and e...
   [FORMAT SCORE]: 30.0

[MODEL]: To address the scheduling conflict and ensure seamless management of both social commitments and har...
   [FORMAT SCORE]: 30.0

[MODEL]: The target is to quickly address the flight cancellation conflict to minimize inconvenience. However...
   [FORMAT SCORE]: 30.0

[MODEL]: In this scenario, the primary goal is to ensure that all parties have a positive, satisfying experie...
   [FORMAT SCORE]: 30.0

[MODEL]: Conflict resolution requires balancing social obligations with work demands while ensuring deadlines...
   [FORMAT SCORE]: 30.0

[MODEL]: To resolve scheduling conflicts in "Monday from hell," I need to carefully consider the urgency of t...
   [FORMAT SCORE]: 30.0

[MODEL]: To resolve scheduling conflicts in the "travel_chaos" scenario, I need to prioritize social satisfac...
   [FORMAT SCORE]: 30.0

[MODEL]: In the morning crunch sc

KeyboardInterrupt: 

In [49]:
import re
import json

def agent_bridge(model_output):
    # 1. Extract Thought
    thought = re.search(r'<thought>(.*?)</thought>', model_output, re.DOTALL)
    thought_text = thought.group(1).strip() if thought else "No reasoning found."
    
    # 2. Extract Action (Even if it's descriptive JSON)
    # We look for keywords in the model's response to force it into our environment
    if "delegate" in model_output.lower() or "assignee" in model_output.lower():
        command = "delegate"
    elif "reschedule" in model_output.lower() or "new_time" in model_output.lower():
        command = "reschedule"
    elif "cancel" in model_output.lower():
        command = "cancel"
    else:
        command = "do_nothing"
        
    print(f"🧐 BRAIN REASONING: {thought_text}")
    print(f"⚙️ EXECUTION ENGINE: Running '{command}' command...")
    
    return {"command": command, "reasoning": thought_text}

# Test it with your last output
result = agent_bridge(outputs[0]["generated_text"])


🧐 BRAIN REASONING: ...
⚙️ EXECUTION ENGINE: Running 'delegate' command...


In [35]:
# EMERGENCY NATIVE MERGE SCRIPT
print("🔗 Merging adapters using Native HF method...")

# 1. Merge (Isme thoda time lag sakta hai)
merged_model = model.merge_and_unload()

# 2. Push directly to Hub
print("⬆️ Pushing merged model to Hugging Face...")
merged_model.push_to_hub("purvansh01/conflict-env-final", token=hf_token)
tokenizer.push_to_hub("purvansh01/conflict-env-final", token=hf_token)

print("✅ DONE! Everything is on Hugging Face profile now.")


🔗 Merging adapters using Native HF method...
⬆️ Pushing merged model to Hugging Face...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ DONE! Everything is on Hugging Face profile now.


In [70]:
import time
import json
import re

def final_demo_log_v2(scenario):
    print("="*60)
    print(f"📡 INCOMING CONFLICT SCENARIO: \n   '{scenario}'")
    print("="*60)
    
    print("\n🧠 STEP 1: ACTIVATING DEEP REASONING ENGINE (GRPO)...")
    
    # FORCING TOKENS: max_length ko hata diya taaki model chup na ho
    prompt = f"<|im_start|>system\nYou are an Elite Assistant. Resolve conflicts logically.\n<|im_end|>\n<|im_start|>user\n{scenario}\n<|im_end|>\n<|im_start|>assistant\n<thought>\n"
    
    outputs = pipe(
        prompt, 
        max_new_tokens=512, 
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id # Added for safety
    )
    
    # Sirf assistant ka naya part nikalte hain
    raw_output = outputs[0]["generated_text"].split("<|im_start|>assistant")[-1]
    
    # 1. Show Reasoning (Pura text jo assistant ne bola)
    if "</thought>" in raw_output:
        thought_text = raw_output.split("</thought>")[0].replace("<thought>", "").strip()
    else:
        # Agar tag band nahi hua, toh pehla 200 character lelo
        thought_text = raw_output[:300].strip() + "..."
    
    print(f"\n[INTERNAL THOUGHTS]:\n{thought_text}")
    print("-" * 60)
    
    print("\n⚙️ STEP 2: PARSING AGENTIC COMMAND...")
    
    # 2. Extract Action (Brute Force Search)
    action = "RESCHEDULE" # Default safe action
    if "delegate" in raw_output.lower(): action = "DELEGATE"
    elif "cancel" in raw_output.lower(): action = "CANCEL"
    elif "reschedule" in raw_output.lower(): action = "RESCHEDULE"
    
    print(f"DETECTED ACTION: {action}")
    
    print(f"\n✅ STEP 3: ENVIRONMENT UPDATED")
    print(f"   [RESULT]: Conflict resolved via {action} logic.")
    print("="*60)

# Run it
final_demo_log_v2("Wife needs pickup at 5:00 PM. Investor Pitch at 5:15 PM. Team meeting at 4:30 PM.")


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📡 INCOMING CONFLICT SCENARIO: 
   'Wife needs pickup at 5:00 PM. Investor Pitch at 5:15 PM. Team meeting at 4:30 PM.'

🧠 STEP 1: ACTIVATING DEEP REASONING ENGINE (GRPO)...

[INTERNAL THOUGHTS]:
<thought>
To resolve this conflict, I need to prioritize tasks based on their importance and urgency.

Step 1: Identify the priorities:
- Wife's pickup (highest priority)
- Investor pitch (second highest priority)
- Team meeting (lowest priority)

Step 2: Determine which tasks can be adjusted or po...
------------------------------------------------------------

⚙️ STEP 2: PARSING AGENTIC COMMAND...
DETECTED ACTION: RESCHEDULE

✅ STEP 3: ENVIRONMENT UPDATED
   [RESULT]: Conflict resolved via RESCHEDULE logic.
